In [1]:
import polars as pl
import pandas as pd

# --- Конфигурация ---
USERS_FILE = "../data/eval_users.csv"
EVENTS_FILE = "../data/eval_user_events.pq"
ITEMS_FILE = "../data/item_features.parquet"
OUTPUT_FILE = "../data/user_description.csv"

In [2]:
# --- Шаг 1: Загружаем пользователей ---
users_df = pl.read_csv(USERS_FILE)
print(f"Users: {len(users_df)}")

# --- Шаг 2: Загружаем события (только нужные колонки) ---
events_cols = ["user_id", "eid", "item_id"]
events_df = pl.read_parquet(EVENTS_FILE, columns=events_cols)
print(f"Events: {len(events_df)}")

# --- Шаг 3: Загружаем признаки items (только нужные колонки) ---
items_cols = ["item_id", "vertical_id", "category_ext_y", "region_id_y", "loc_id_y"]
items_df = pl.read_parquet(ITEMS_FILE, columns=items_cols)
print(f"Items: {len(items_df)}")

# --- Шаг 4: Join events + items по item_id ---
merged = events_df.join(items_df, on="item_id", how="left")
print(f"Merged: {len(merged)}")

# Удаляем item_id — больше не нужен
merged = merged.drop("item_id")

# --- Шаг 5: Для каждого user_id находим mode по каждому столбцу ---
agg_cols = ["eid", "vertical_id", "category_ext_y", "region_id_y", "loc_id_y"]

result = (
    merged
    .group_by("user_id")
    .agg(
        [pl.col(col).mode().first().alias(col) for col in agg_cols]
    )
)

# --- Шаг 6: Join с users чтобы гарантировать всех пользователей ---
final = users_df.join(result, on="user_id", how="left")

print(f"Final shape: {final.shape}")
print(final.head())

# --- Шаг 7: Сохраняем ---
final.write_csv(OUTPUT_FILE)
print(f"Saved to {OUTPUT_FILE}")

Users: 94408
Events: 96807172
Items: 178327659
Merged: 96807172
Final shape: (94408, 6)
shape: (5, 6)
┌─────────┬─────┬─────────────┬────────────────┬─────────────┬──────────┐
│ user_id ┆ eid ┆ vertical_id ┆ category_ext_y ┆ region_id_y ┆ loc_id_y │
│ ---     ┆ --- ┆ ---         ┆ ---            ┆ ---         ┆ ---      │
│ i64     ┆ u32 ┆ u32         ┆ i64            ┆ i64         ┆ i64      │
╞═════════╪═════╪═════════════╪════════════════╪═════════════╪══════════╡
│ 33      ┆ 7   ┆ 0           ┆ 28             ┆ 53          ┆ 2368     │
│ 63      ┆ 7   ┆ 0           ┆ 5              ┆ 28          ┆ 1165     │
│ 188     ┆ 7   ┆ 0           ┆ 1              ┆ 44          ┆ 1941     │
│ 281     ┆ 7   ┆ 2           ┆ 46             ┆ 2           ┆ 39       │
│ 295     ┆ 7   ┆ 3           ┆ 8              ┆ 25          ┆ 824      │
└─────────┴─────┴─────────────┴────────────────┴─────────────┴──────────┘
Saved to ../data/user_description.csv
